# CloudBQ — deploy scenario A from a notebook and witness the SQL

The same sequence the Cloud Shell runbook executes (`docs/PASS1_SEQUENCE.md`), one cell per step, on top of the same
`cloud_function_new_deals/deploy.sh`. Nothing is re-implemented: a cell that reads `sh(f"bash {D} tables")` runs exactly
what `$D tables` runs in Cloud Shell. What the notebook adds is the **witness** part: after the function is deployed, the
schema, the hot-score UDF, the golden vectors and the audit tables are executed through the BigQuery Python client, with the
job id, bytes processed and slot time of every statement visible next to its SQL, and the Python scorer (`main.hot_score`)
is compared with the SQL scorer on the same inputs. The `enable` step comes **after** the witness section: look, then schedule.

| Cloud Shell | Notebook cell |
|---|---|
| `gcloud cloud-shell ssh --authorize-session` | run the notebook on a Vertex AI Workbench instance in the same project (see `notebooks/README.md`) |
| `gcloud config set project …` | the settings cell exports `CLOUDSDK_CORE_PROJECT` |
| `git clone … && git checkout … && git pull` | **1. Repository** |
| `cp .env.example .env && nano .env` | **2. Settings** (edits `SETTINGS`, writes `.env`, refuses placeholders like `check-env`) |
| `DRYRUN=1 $D all` … `$D probe` | **3. Sequence** — one cell per step, in the runbook order; a failing step stops Run-All (the gate) |
| `bq query …` by hand | **4. Witness** — the SQL from the repo, run from Python, with execution metadata |
| `$D enable` | **5. Enable** — flag-guarded; the two Scheduler jobs stay paused until you flip it |

**Identity — two credential stores.** `gcloud` and `bq` (so every `deploy.sh` step) act as gcloud's *active account*; the
BigQuery and Secret Manager Python clients act as *Application Default Credentials* (ADC). On a Workbench instance both are the
instance's service account (the Compute Engine default one unless the instance was created with `--service-account-email`),
which usually cannot enable APIs, create service accounts or deploy Cloud Run. Pick one route before step 3:

- **act as yourself**: in a JupyterLab *Terminal* run `gcloud auth login --no-launch-browser --update-adc` once. `--update-adc`
  is what couples the two stores; `gcloud auth application-default login` alone would leave `deploy.sh` acting as the service account.
- **act as the instance's service account**: grant it what the Cloud Shell operator has — Owner on a sandbox project, or Cloud Run
  Admin, Cloud Run Source Developer, Service Account Admin, Service Account User, Project IAM Admin, Service Usage Admin, Secret
  Manager Admin, BigQuery Admin, Cloud Scheduler Admin, Cloud Build Editor; `deploy.sh` mints the service's ID token with the right
  audience for a service account, so `dry` and `run` work on this route too.

The identity cell shows both stores.

## 0. Helper — run a shell command like a terminal would

In [ ]:
import os, re, sys, json, shlex, subprocess, pathlib, getpass
from datetime import datetime, timedelta, timezone

def sh(cmd, env=None, check=True):
    """Stream a shell command's output into the cell; raise when it fails so Run-All stops where deploy.sh stops.
    CLOUDSDK_CORE_DISABLE_PROMPTS=1 answers gcloud's interactive questions with their defaults (a notebook has no TTY):
    the first `gcloud run deploy --source` then creates the Artifact Registry repository instead of asking."""
    full_env = {**os.environ, "CLOUDSDK_CORE_DISABLE_PROMPTS": "1", **(env or {})}
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, env=full_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p.returncode

def out(cmd):
    """Capture a command's stdout (no streaming)."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

## 1. Repository — clone or update the working branch

In [ ]:
REPO_URL = "https://github.com/Wkayaobama/CloudBQ.git"
BRANCH = "claude/cloud-run-bigquery-pipeline-e94ziy"
_cwd = pathlib.Path.cwd()
# opened from inside a clone (notebooks/ of the repo) → use that clone; otherwise ~/CloudBQ (the Workbench boot disk persists it)
REPO = _cwd.parent if (_cwd.name == "notebooks" and (_cwd.parent / ".git").exists()) else pathlib.Path.home() / "CloudBQ"
if not (REPO / ".git").exists():
    sh(f"git clone --branch {shlex.quote(BRANCH)} {REPO_URL} {shlex.quote(str(REPO))}")
os.chdir(REPO)
sh(f"git fetch -q origin {BRANCH} && git checkout -q {BRANCH}")
dirty = out("git status --porcelain")
if dirty:   # this notebook, once edited or saved with outputs, is a local change; a pull that would overwrite it is refused by git
    print("local changes — not pulling (commit or stash them to update the clone):\n" + dirty)
else:
    sh(f"git pull -q --ff-only origin {BRANCH}")
sh("git log --oneline -1")
D = "cloud_function_new_deals/deploy.sh"        # the one execution surface of scenario A
print("repo:", REPO)

## 2. Settings — `.env` from `.env.example`

Edit `SETTINGS` (at least `PROJECT`) and run. Keys whose value is not a placeholder are written into `.env` (created from
`.env.example` when missing); keys left as placeholders leave `.env` untouched, so a hand-edited `.env` survives. The file is
gitignored; like `check-env`, the cell refuses placeholders. `BQ_LOCATION` must be the location of an *existing* `hubspot`
dataset if there is one (`bq show --format=prettyjson PROJECT:hubspot | grep location`); `CREATE SCHEMA IF NOT EXISTS` will not move it.

In [ ]:
ENV_FILE = REPO / ".env"
SETTINGS = {                       # ← edit, then run; everything else keeps the defaults of .env.example
    "PROJECT": "your-gcp-project-id",
    "REGION": "europe-west6",
    "BQ_LOCATION": "EU",
    "DATASET": "hubspot",
    "FN": "new-deals-report",
    "SA_NAME": "new-deals-fn",
    "PORTAL_ID": "9201667",
}
def is_placeholder(v): return (not v) or v.startswith(("your-", "your_", "<"))

text = (ENV_FILE if ENV_FILE.exists() else REPO / ".env.example").read_text()
for k, v in SETTINGS.items():
    if not is_placeholder(v):
        text, n = re.subn(rf"^{k}=.*$", f"{k}={v}", text, flags=re.M)
        if n == 0: text += f"\n{k}={v}\n"
ENV_FILE.write_text(text)
print(f"{ENV_FILE} {'updated' if ENV_FILE.exists() else 'written'} from SETTINGS")

def load_env(path):
    """Read .env the way deploy.sh does (`. .env`): shell quoting, `#` comments after values."""
    env = {}
    for line in path.read_text().splitlines():
        parts = shlex.split(line, comments=True)
        if parts and "=" in parts[0]:
            k, v = parts[0].split("=", 1); env[k] = v
    return env

ENV = load_env(ENV_FILE)
for k in ("PROJECT", "REGION", "DATASET", "FN", "SA_NAME", "BQ_LOCATION"):
    assert not is_placeholder(ENV.get(k, "")), f"{k} is unset or still the placeholder — edit SETTINGS above (or {ENV_FILE}) and re-run"
assert not ENV.get("HUBSPOT_TOKEN"), "HUBSPOT_TOKEN must not be in .env (the secret cell prompts for it)"
PROJECT, REGION, BQ_LOCATION, DATASET, FN, SA_NAME = (ENV[k] for k in ("PROJECT", "REGION", "BQ_LOCATION", "DATASET", "FN", "SA_NAME"))
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT   # what `gcloud config set project` does in Cloud Shell
print({k: ENV.get(k) for k in ("PROJECT", "REGION", "BQ_LOCATION", "DATASET", "FN", "SA_NAME", "PORTAL_ID", "SECRET_VERSION")})

### Identity — the two credential stores the notebook acts with

In [ ]:
print("gcloud / bq / deploy.sh act as the active account:", out("gcloud auth list --filter status:ACTIVE --format 'value(account)' 2>/dev/null") or "(none — gcloud is not authenticated)")
import google.auth, google.auth.transport.requests
try:
    creds, adc_project = google.auth.default()
    if hasattr(creds, "service_account_email"):                 # compute credentials learn their email on the first refresh
        try: creds.refresh(google.auth.transport.requests.Request())
        except Exception: pass
    who = getattr(creds, "service_account_email", None) or ("user credentials" if "oauth2" in type(creds).__module__ else type(creds).__name__)
    print(f"BigQuery / Secret Manager clients act as Application Default Credentials: {who} | quota project: {adc_project}")
except Exception as e:                                           # no ADC at all: the first client call below would fail with the same reason
    print(f"No Application Default Credentials: {e}")
print("Both should be you (gcloud auth login --update-adc) or both the instance's service account with operator roles — see the top of the notebook.")

## 3. The sequence — the Cloud Shell runbook, one cell per step

Same script, same order. Every step is create-or-update, so a cell can be re-run after a fix. A failing step raises and
stops Run-All: that is the gate semantics of `deploy.sh`.

### 3.0 Rehearsal — `DRYRUN=1 $D all` prints every gcloud/bq command and runs nothing

In [ ]:
sh(f"DRYRUN=1 bash {D} all")

### 3.1 `check-env` — `.env` complete, no placeholders, no token in it, gcloud and bq present

In [ ]:
sh(f"bash {D} check-env")

### 3.2 `enable-apis` — run, cloudbuild, artifactregistry, secretmanager, cloudscheduler, bigquery

In [ ]:
sh(f"bash {D} enable-apis")

### 3.3 `iam` — runtime service account, `bigquery.jobUser`, `run.builder` for the Cloud Build service account (IAM changes take a couple of minutes to propagate; if `deploy` fails on permissions right after this, re-run it)

In [ ]:
sh(f"bash {D} iam")

### 3.4 `tables` — `CREATE SCHEMA IF NOT EXISTS`, `report_state`, `new_deals`, `new_deals_runs`, `new_deals_history`, the view, the `dataEditor` GRANT

In [ ]:
sh(f"bash {D} tables")

### 3.5 `secret` — the HubSpot private-app token into Secret Manager

The Cloud Shell line is `HUBSPOT_TOKEN=... $D secret`. Here the token is typed into a masked prompt and handed to the step as
an environment variable only; it is never written to the notebook or to `.env`. Every run of this step adds a new secret
version; the step prints the version number, which `deploy` uses unless `.env` pins `SECRET_VERSION`.

In [ ]:
token = getpass.getpass("HUBSPOT_TOKEN of the private app (crm.objects.deals.read, crm.schemas.deals.read, crm.objects.owners.read): ")
try:
    sh(f"bash {D} secret", env={"HUBSPOT_TOKEN": token})
finally:
    del token

### 3.6 `deploy` — canonical `gcloud run deploy $FN --source … --function new_deals_report --base-image python312`, private, one instance, one request at a time (a source build takes a few minutes; the first one creates the `cloud-run-source-deploy` Artifact Registry repository)

In [ ]:
sh(f"bash {D} deploy")

### 3.7 `invoker` — `run.invoker` for the service account; the two Scheduler jobs are created **paused**

In [ ]:
sh(f"bash {D} invoker")

### 3.8 `dry` — the gate

`?dry_run=1` for both entities: the function reads the watermark and the warehouse twins exactly like a live run and writes
nothing, not even the audit row. The step asserts `rows_written == 0`; an HTTP error (403: the caller lacks `run.invoker`;
409: a lease is held) fails the step too. A failure raises here and nothing below should run.

In [ ]:
sh(f"bash {D} dry")

### 3.9 `run SEALSQ`, `run ICALPS` — the first live window per entity (no watermark yet → `FIRST_RUN_DAYS` = 7 days back); each run moves the watermark, so a second `run` covers the next window (`replay` repeats one)

In [ ]:
sh(f"bash {D} run SEALSQ")

In [ ]:
sh(f"bash {D} run ICALPS")

### 3.10 `probe` — what landed, and the last runs (the same two queries are run from Python in 4.5 — identical results are the equality check)

In [ ]:
sh(f"bash {D} probe")

## 4. Witness — the repo's SQL executed from Python, with its execution metadata

`q()` prints the SQL, runs it through the BigQuery client, then prints the job id, statement type, bytes processed, slot time
and cache flag before returning the rows as a DataFrame. `render()` performs the same `${PROJECT}`/`${DATASET}`/`${BQ_LOCATION}`
substitution that `deploy.sh tables` performs with `envsubst`.

In [ ]:
sh(f"{shlex.quote(sys.executable)} -m pip install -q -r cloud_function_new_deals/requirements.txt db-dtypes pandas")   # into THIS kernel: main.py imports functions_framework; db-dtypes for to_dataframe()
from google.cloud import bigquery
from IPython.display import display
import pandas as pd
client = bigquery.Client(project=PROJECT, location=BQ_LOCATION)
TBL = f"`{PROJECT}.{DATASET}"           # prefix: TBL + ".name`"

def render(path):
    return (pathlib.Path(path).read_text()
            .replace("${PROJECT}", PROJECT).replace("${DATASET}", DATASET).replace("${BQ_LOCATION}", BQ_LOCATION))

def q(sql, show=True, **params):
    """Run SQL in BigQuery and show what happened, then return the rows as a DataFrame."""
    if show:
        print(sql.strip()); print()
    types = {str: "STRING", int: "INT64", float: "FLOAT64", bool: "BOOL", datetime: "TIMESTAMP"}
    cfg = bigquery.QueryJobConfig(query_parameters=[bigquery.ScalarQueryParameter(k, types[type(v)], v) for k, v in params.items()]) if params else None
    job = client.query(sql, job_config=cfg)
    rows = job.result()
    took = (job.ended - job.started).total_seconds() if job.ended and job.started else None
    print(f"job {job.job_id} | {job.statement_type} | bytes processed {job.total_bytes_processed} | slot ms {job.slot_millis} | cache hit {job.cache_hit} | {took}s"
          + (f" | DDL {job.ddl_operation_performed} on {job.ddl_target_table or job.ddl_target_routine}" if getattr(job, "ddl_operation_performed", None) else ""))
    return rows.to_dataframe()

def all_true(series):
    """A NULL in a BOOL column must fail, not be skipped by pandas' nullable .all()."""
    return bool(series.notna().all() and series.astype(bool).all())

### 4.1 The schema `tables` created — `INFORMATION_SCHEMA` and the view's text

In [ ]:
display(q(f"SELECT table_name, table_type, creation_time FROM {TBL}.INFORMATION_SCHEMA.TABLES` ORDER BY table_name"))
display(q(f"SELECT table_name, column_name, data_type FROM {TBL}.INFORMATION_SCHEMA.COLUMNS` WHERE table_name = 'new_deals' ORDER BY ordinal_position", show=False))
display(q(f"SELECT table_name, view_definition FROM {TBL}.INFORMATION_SCHEMA.VIEWS` WHERE table_name = 'new_deals_last_7d'", show=False))

### 4.2 The hot score as SQL — create `hot_score_v1` from `scoring/hot_score_v1.sql`, prove it on the golden vectors

`scoring/hot_score_v1.sql` is the BigQuery port of `main.hot_score` (scenario C's compute lane uses it; here it is the witness).
`tests/hot_score_vectors.sql` is generated from `tests/hot_score_vectors.json` by `tests/test_hot_score.py`; every row must show `ok = TRUE`.

In [ ]:
display(q(render("scoring/hot_score_v1.sql")))                       # CREATE OR REPLACE FUNCTION … (DDL; the function body is the SQL you see)
vec = q(render("tests/hot_score_vectors.sql"), show=False)          # 15 vectors through the UDF
display(vec)
assert all_true(vec["ok"]), vec[vec["ok"] != True]
print("SQL UDF reproduces all", len(vec), "golden vectors")

### 4.3 The same vectors through the Python scorer the function runs — side by side with SQL

In [ ]:
sys.path.insert(0, str(REPO / "cloud_function_new_deals")); sys.path.insert(0, str(REPO / "tests"))
import main                      # the function's module: hot_score(), HubSpot client, shape()
import test_hot_score as ths     # emit_sql(): the vector → SQL check emitter (reused on live deals below)
vectors = json.loads((REPO / "tests" / "hot_score_vectors.json").read_text())
NOW = datetime.fromisoformat(vectors["now"].replace("Z", "+00:00"))
py = pd.DataFrame([{"id": c["id"], "py_score": main.hot_score(c["props"], NOW)[0], "py_hot": main.hot_score(c["props"], NOW)[1]} for c in vectors["cases"]])
display(py)

In [ ]:
cmp = vec.merge(py, on="id", how="outer")

def same(r):   # not-scored pipelines are NULL on both sides; scored ones agree within the 0.1 rounding tolerance and on the flag
    if pd.isna(r["score"]) or pd.isna(r["py_score"]):
        return bool(pd.isna(r["score"]) and pd.isna(r["py_score"]) and pd.isna(r["hot"]) and pd.isna(r["py_hot"]))
    return abs(float(r["score"]) - float(r["py_score"])) <= 0.1 and bool(r["hot"]) == bool(r["py_hot"])

cmp["same"] = cmp.apply(same, axis=1)
display(cmp[["id", "py_score", "score", "py_hot", "hot", "same"]])
assert cmp["same"].all(), cmp[~cmp["same"]]
print("Python and SQL agree on every vector")

### 4.4 What the function wrote — state, runs, history, the weekly view

`report_state` holds the watermark and the lease per entity; `new_deals_runs` one row per live run (dry runs write none);
`new_deals_history` the per-deal scores per run; `new_deals_last_7d` is what Breeze / the delivery read.

In [ ]:
display(q(f"SELECT report_id, watermark, lock_until, updated_at FROM {TBL}.report_state` ORDER BY report_id"))
display(q(f"SELECT run_ts, entity, mode, dry_run, orchestrator, row_count, rows_written, duplicate_suspects, day1_hot, status, duration_ms FROM {TBL}.new_deals_runs` ORDER BY run_ts DESC LIMIT 10", show=False))
display(q(f"SELECT run_ts, entity, COUNT(*) AS deals, COUNTIF(day1_hot_flag) AS hot FROM {TBL}.new_deals_history` GROUP BY run_ts, entity ORDER BY run_ts DESC LIMIT 10", show=False))
display(q(f"SELECT entity, deal_id, dealname, pipeline, stage, probability_pct, amount_kusd, day1_hot_score, day1_hot_flag, duplicate_suspect, createdate FROM {TBL}.new_deals_last_7d` LIMIT 50", show=False))

### 4.5 Equality check — `deploy.sh probe` from the shell and the same two queries from Python

The script's comment says it: *run from any surface; identical result is the equality check.*

In [ ]:
sh(f"bash {D} probe")
display(q(f"SELECT entity, COUNT(*) n, COUNTIF(day1_hot_flag) hot, COUNTIF(duplicate_suspect) dup, MIN(createdate) oldest, MAX(createdate) newest FROM {TBL}.new_deals` GROUP BY entity ORDER BY entity", show=False))
display(q(f"SELECT run_ts, entity, mode, dry_run, orchestrator, row_count, rows_written, status FROM {TBL}.new_deals_runs` ORDER BY run_ts DESC LIMIT 10", show=False))

### 4.6 The function's own dry payload, from Python

Calls the private service with an ID token and shows the payload `deploy.sh dry` only summarises: the window, the per-row
`day1_hot_score`, the `day1_hot` list. The token follows the ADC type: user credentials → `gcloud auth print-identity-token`
(accepted by Cloud Run as is); a service account → a token minted by the metadata server with the service URL as audience.
The identity needs `run.invoker` on the service (an Owner has it; the instance's service account only if you granted it).

In [ ]:
import requests, google.oauth2.id_token, google.oauth2.credentials
URL = out(f"bash {D} url").splitlines()[-1]

def id_token_for(audience):
    creds, _ = google.auth.default()
    if isinstance(creds, google.oauth2.credentials.Credentials):          # a person (gcloud auth login --update-adc)
        return out("gcloud auth print-identity-token")
    return google.oauth2.id_token.fetch_id_token(google.auth.transport.requests.Request(), audience)   # service account: audience = service URL

payloads = {}
for entity in ("SEALSQ", "ICALPS"):
    r = requests.get(f"{URL}?entity={entity}&dry_run=1", headers={"Authorization": f"Bearer {id_token_for(URL)}"}, timeout=600)
    assert r.status_code == 200, f"{entity}: HTTP {r.status_code} — {r.text[:300]}"
    p = payloads[entity] = r.json()
    assert p["dry_run"] is True and p["rows_written"] == 0, p
    print(entity, {k: p.get(k) for k in ("report_id", "run_timestamp", "orchestrator", "dry_run", "row_count", "rows_written", "duplicate_suspects")}, p.get("window"))
    if p.get("rows"):
        display(pd.DataFrame(p["rows"])[["deal_id", "dealname", "pipeline", "stage", "probability_pct", "amount_kusd", "day1_hot_score", "day1_hot_flag", "duplicate_suspect", "createdate"]])

### 4.7 Optional — live parity: HubSpot deals of the last 7 days scored in Python and in SQL

Needs `secretmanager.secretAccessor` on `hubspot-token` for the notebook's ADC identity (the `secret` step granted it to the
function's service account only; an Owner has it). The deals are fetched with the function's own `HubSpot` client, scored with
`main.hot_score`, and the pairs are emitted as the same SQL check the golden vectors use (`test_hot_score.emit_sql`, typed NULLs).

In [ ]:
try:
    from google.cloud import secretmanager
except ImportError:
    sh(f"{shlex.quote(sys.executable)} -m pip install -q google-cloud-secret-manager")
    import importlib; importlib.invalidate_caches()        # google.cloud is a namespace package: refresh it before re-importing
    from google.cloud import secretmanager
sm = secretmanager.SecretManagerServiceClient()
hs = main.HubSpot(sm.access_secret_version(name=f"projects/{PROJECT}/secrets/hubspot-token/versions/latest").payload.data.decode())
end = datetime.now(timezone.utc).replace(microsecond=0); start = end - timedelta(days=7)
deals = [d for d in hs.search_created(start, end) if d.get("properties", {}).get("pipeline") in main.CR]   # the two scored pipelines
print(f"{len(deals)} deals created {start:%Y-%m-%d} → {end:%Y-%m-%d} in scored pipelines")
live = {"now": end.strftime("%Y-%m-%dT%H:%M:%SZ"), "cases": []}
for d in deals:
    p = d["properties"]; score, hot = main.hot_score(p, end)
    live["cases"].append({"id": re.sub(r"[^A-Za-z0-9 ]", " ", f"{d['id']} {p.get('dealname') or ''}")[:60], "props": p, "expected": {"score": score, "hot": hot}})
if live["cases"]:
    parity = q(ths.emit_sql(live).replace("${PROJECT}", PROJECT).replace("${DATASET}", DATASET), show=False)
    display(parity)
    assert all_true(parity["ok"]), parity[parity["ok"] != True]
    print("Python (the function) and SQL (the UDF) agree on every live deal")

## 5. Enable — resume the two Scheduler jobs (Mon 07:00 and 07:20 Europe/Zurich)

The last step of the runbook. Flip the flag only after the probe and the witness section looked right. `enable` is idempotent:
a job that is already enabled is reported, not resumed again.

In [ ]:
I_HAVE_READ_THE_PROBE_AND_WANT_THE_WEEKLY_SCHEDULE = False
assert I_HAVE_READ_THE_PROBE_AND_WANT_THE_WEEKLY_SCHEDULE, "flip the flag above after the probe looked right — this resumes the two Scheduler jobs"
sh(f"bash {D} enable")
sh(f"bash {D} status")

## 6. Operate

`status` shows watermarks, leases and the two jobs; `pause` stops the schedule; `replay ENTITY FROM_ISO` rewinds a watermark and
runs (MERGE on `deal_id` keeps it idempotent); `logs` tails the function.

In [ ]:
sh(f"bash {D} status")
# sh(f"bash {D} pause")
# sh(f"bash {D} replay SEALSQ 2026-10-01T00:00:00Z")
# sh(f"bash {D} logs")